# Gold | Validação cruzada: taxa ponderada (microdados) × taxa oficial (agregado do INEP)
Compara `ft_taxa_alfabetizacao_ponderada` com `ft_resultado_municipio`. Diferenças são **informativas**: indicam que a fórmula
ou os códigos de presença/alfabetizado (widgets do notebook de fatos) precisam ser conferidos no dicionário. Não é uma regra bloqueante.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
dbutils.widgets.text("tolerancia_pp", "1.0", "Tolerância (pontos percentuais)")
TOL = float(dbutils.widgets.get("tolerancia_pp"))
_m = iniciar_etapa()
dq = RegistroDQ("GOLD")
cmp = spark.sql("""
    SELECT p.ano, p.id_municipio, p.serie, p.rede_codigo,
           p.taxa_alfabetizacao_ponderada AS taxa_microdados, o.taxa_alfabetizacao AS taxa_oficial,
           ABS(p.taxa_alfabetizacao_ponderada - o.taxa_alfabetizacao) AS diff_pp
    FROM workspace.gold_staging.ft_taxa_alfabetizacao_ponderada p
    JOIN workspace.gold_staging.ft_resultado_municipio o
      ON p.ano = o.ano AND p.id_municipio = o.id_municipio AND p.serie = o.serie AND p.rede_codigo = o.rede_codigo
    WHERE p.taxa_alfabetizacao_ponderada IS NOT NULL AND o.taxa_alfabetizacao IS NOT NULL""")
total = cmp.count()
fora = cmp.filter(f"diff_pp > {TOL}").count()
dq.registrar("ft_taxa_alfabetizacao_ponderada x ft_resultado_municipio", "DIVERGENCIA_TAXA_OFICIAL", "INFORMATIVA", fora,
             f"{fora}/{total} recortes divergem mais de {TOL} p.p. da taxa oficial. Conferir códigos de presenca/alfabetizado.")
display(cmp.orderBy("diff_pp", ascending=False).limit(50))
registrar_execucao("GOLD", "validacao_cruzada", _m, row_count=total)